<a href="https://colab.research.google.com/github/uw-chris/GB885-Final-Project-Sheng-C/blob/main/GB885_Final_Project_Sheng_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Select and Load the Data
# ============================================
import pandas as pd
import numpy as np

# locate files in directory
from google.colab import files
uploaded = files.upload()

# load data
sales = pd.read_csv('TABLE_SALES_885.csv')
retailer = pd.read_csv('TABLE_RETAILER_885.csv')
product = pd.read_csv('TABLE_PRODUCTS_885.csv', sep='|')

# show
print("SALES:", sales.shape)
print("RETAILER:", retailer.shape)
print("PRODUCT:", product.shape)

# evalute structure
sales.info()
retailer.info()
product.info()

# table review
display(sales.head())
display(retailer.head())
display(product.head())

# data check
print("\nMissing values (SALES):\n", sales.isna().sum())
print("\nUnique SALES_METHOD values:", sales['SALES_METHOD'].unique())
print("\nUnique UNITS_SOLD dtype issue check:", sales['UNITS_SOLD'].apply(type).value_counts())
print("\nPRICE_PER_UNIT range:", sales['PRICE_PER_UNIT'].min(), "-", sales['PRICE_PER_UNIT'].max())
print("\nRETAILER_IDs in SALES not found in RETAILER table:",
      sales.loc[~sales['RETAILER_ID'].isin(retailer['RETAILER_ID']), 'RETAILER_ID'].tolist())

In [ ]:
# ============================================
# Cleanse Data
# ============================================

# work on a copy so raw data stays untouched for reference
sales_clean = sales.copy()

# UNITS_SOLD convered to numbers (currently stored as object)
# replace "***" text with an NaN values
sales_clean['UNITS_SOLD'] = sales_clean['UNITS_SOLD'].replace('***', np.nan)

# convert column to float (originally object)
sales_clean['UNITS_SOLD'] = sales_clean['UNITS_SOLD'].astype(float)

# SALES_METHOD typo corrections
sales_clean['SALES_METHOD'] = sales_clean['SALES_METHOD'].replace('Ootlet', 'Outlet')
print("Cleaned SALES_METHOD values:", sales_clean['SALES_METHOD'].unique())

## drop rows with missing UNITS_SOLD or PRICE_PER_UNIT
print("Rows before dropping missing values:", len(sales_clean))
sales_clean = sales_clean.dropna(subset=['UNITS_SOLD', 'PRICE_PER_UNIT'])
print("Rows after:", len(sales_clean))

# drop the price outlier (99999 is not a real price)
sales_clean = sales_clean[sales_clean['PRICE_PER_UNIT'] < 10000]

# drop the row with an invalid retailer ID
valid_retailer_ids = retailer['RETAILER_ID']
sales_clean = sales_clean[sales_clean['RETAILER_ID'] != '999999999']

print("Final row count after cleaning:", len(sales_clean))

# correct dtypes now that the bad values are gone
sales_clean['UNITS_SOLD'] = sales_clean['UNITS_SOLD'].astype(int)
sales_clean['INVOICE_DATE'] = pd.to_datetime(sales_clean['INVOICE_DATE'])

# derive REVENUE, since the raw data doesn't include it
sales_clean['REVENUE'] = sales_clean['PRICE_PER_UNIT'] * sales_clean['UNITS_SOLD']

sales_clean.head()

In [ ]:
# ============================================
# Merge Tables
# ============================================

# merge SALES with RETAILER on RETAILER_ID
merged = sales_clean.merge(retailer, on='RETAILER_ID', how='left')

# merge that result with PRODUCT on PRODUCT_ID
merged = merged.merge(product, on='PRODUCT_ID', how='left')

# check data that no new nulls have come in
print("Rows before merging:", len(sales_clean))
print("Rows after merging:", len(merged))
print("\nAny missing RETAILER or PRODUCT_NAME after merge?")
print(merged[['RETAILER', 'PRODUCT_NAME']].isna().sum())

merged.head()

In [ ]:
# ============================================
# Answer Business Questions
# ============================================

# Q1: Which product category had the highest DOLLAR sales in 2021?
sales_2021 = merged[merged['YEAR'] == 2021]

revenue_by_product = sales_2021.groupby('PRODUCT_NAME')['REVENUE'].sum().sort_values(ascending=False)
print("Q1: Revenue by product category, 2021")
print(revenue_by_product)
print("\nTop product:", revenue_by_product.index[0], "-> $", round(revenue_by_product.iloc[0], 2))

In [ ]:
# Q2: Which state had the highest DOLLAR sales of WOMEN'S products in 2021?
womens_2021 = sales_2021[sales_2021['PRODUCT_NAME'].str.contains("Women's")]

revenue_by_state_womens = womens_2021.groupby('STATE')['REVENUE'].sum().sort_values(ascending=False)
print("Q2: Top state, women's products, 2021")
print("Top state:", revenue_by_state_womens.index[0], "-> $", round(revenue_by_state_womens.iloc[0], 2))

In [ ]:
# Q3: Which state had the highest DOLLAR sales of MEN'S products in 2021?
mens_2021 = sales_2021[sales_2021['PRODUCT_NAME'].str.contains("Men's") & ~sales_2021['PRODUCT_NAME'].str.contains("Women's")]

revenue_by_state_mens = mens_2021.groupby('STATE')['REVENUE'].sum().sort_values(ascending=False)
print("Q3: Top state, men's products, 2021")
print("Top state:", revenue_by_state_mens.index[0], "-> $", round(revenue_by_state_mens.iloc[0], 2))

In [ ]:
# Q4: Which retailer purchased the most UNITS in 2021? In 2020?
units_by_retailer_2021 = merged[merged['YEAR'] == 2021].groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)
units_by_retailer_2020 = merged[merged['YEAR'] == 2020].groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)

print("Q4 (2021): Top retailer by units")
print("Top retailer:", units_by_retailer_2021.index[0], "->", units_by_retailer_2021.iloc[0], "units")

print("\nQ4 (2020): Top retailer by units")
print("Top retailer:", units_by_retailer_2020.index[0], "->", units_by_retailer_2020.iloc[0], "units")